# Reasoning LLMs & Leaderboards

Explores **reasoning-focused language models**, memory-efficient **quantization**, and how models are compared on public **benchmark leaderboards**.

**What it demonstrates**
- Loading and running models with quantization (BitsAndBytes) to fit limited GPU memory
- Working with evaluation datasets
- Understanding how leaderboard benchmarks measure model capability

**Stack:** Python · HuggingFace Transformers · datasets · BitsAndBytes


In [1]:
import os
from huggingface_hub import notebook_login
notebook_login()
print("Login successful (or token already present)!")

Login successful (or token already present)!


In [2]:
#!pip uninstall datasets -y
#!pip install datasets pyarrow fsspec multiprocess

In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import gradio as gr
from IPython.display import display, Markdown
import random

print("All imports successful!")

All imports successful!


In [3]:
from datasets import load_dataset

In [4]:
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    torch.set_default_device("cuda")
    print("PyTorch default device set to CUDA (GPU).")
else:
    print("WARNING: No GPU detected. Performance will be very slow.")
    print("Go to Runtime > Change runtime type and select GPU.")

GPU detected: NVIDIA GeForce RTX 4080 Laptop GPU
PyTorch default device set to CUDA (GPU).


In [5]:
# Helper function for markdown display
def print_markdown(text):
    """Displays text as Markdown."""
    display(Markdown(text))

In [6]:
dataset_id = "britneymuller/cnbc_newsfeed"

print(f"Loading dataset '{dataset_id}'...")

new_dataset = load_dataset(dataset_id, split="train")

print("Dataset loaded successfully!")

Loading dataset 'britneymuller/cnbc_newsfeed'...
Dataset loaded successfully!


In [7]:
new_dataset.to_pandas().head()

,title,url,published_at,author,publisher,short_description,keywords,header_image,raw_description,description,scraped_at
0,Santoli’s Wednesday market notes: Could Septem...,https://www.cnbc.com/2021/09/29/santolis-wedne...,2021-09-29T17:09:39+0000,Michael Santoli,CNBC,"This is the daily notebook of Mike Santoli, CN...","cnbc, Premium, Articles, Investment strategy, ...",https://image.cnbcfm.com/api/v1/image/10694960...,"<div class=""group""><p><em>This is the daily no...","This is the daily notebook of Mike Santoli, CN...",2021-10-30 14:11:23.709372
1,My take on the early Brexit winners and losers,https://www.cnbc.com/2016/06/24/ian-bremmers-t...,2016-06-24T13:50:48-0400,NaN,CNBC,This commentary originally ran on Facebook. Bo...,"Articles, Politics, Europe News, European Cent...",https://fm.cnbc.com/applications/cnbc.com/reso...,NaN,NaN,2021-10-30 14:11:23.820139
2,Europe&#039;s recovery depends on Renzi&#039;s...,https://www.cnbc.com/2014/03/25/europes-recove...,2014-03-25T13:29:45-0400,NaN,CNBC,"In spring, ambitious reforms began in Italy. U...","Articles, Business News, Economy, Europe Econo...",https://fm.cnbc.com/applications/cnbc.com/reso...,NaN,NaN,2021-10-30 14:11:23.854710
3,US Moves Closer to Becoming A Major Shareholde...,https://www.cnbc.com/2009/04/22/us-moves-close...,2009-04-22T19:49:03+0000,Michelle Caruso-Cabrera,CNBC,The US government is increasingly likely to co...,"cnbc, Articles, General Motors Co, Business Ne...",https://image.cnbcfm.com/api/v1/image/24947979...,"<div class=""group""><p>The US government is inc...",The US government is increasingly likely to co...,2021-10-30 14:11:24.261143
4,Trump: 'Mission accomplished' on 'perfectly ex...,https://www.cnbc.com/2018/04/14/trump-mission-...,2018-04-14T14:59:04+0000,Javier E. David,CNBC,NaN,"cnbc, Articles, George W. Bush, Vladimir Putin...",https://image.cnbcfm.com/api/v1/image/10513177...,"<div class=""group""></div>,<div class=""group""><...",President Donald Trump hailed the U.S.-led int...,2021-10-30 14:11:24.489490


In [8]:
new_dataset.features

{'title': Value('string'),
 'url': Value('string'),
 'published_at': Value('string'),
 'author': Value('string'),
 'publisher': Value('string'),
 'short_description': Value('string'),
 'keywords': Value('string'),
 'header_image': Value('string'),
 'raw_description': Value('string'),
 'description': Value('string'),
 'scraped_at': Value('string')}

In [9]:
def combine_new_text(example):

    title = example.get("title", "") or ""
    description = example.get("description", "") or ""

    return {"full_text": f"Title: {title}\nDescription: {description}"}

In [10]:
new_dataset = new_dataset.map(combine_new_text)

print(new_dataset[99]["full_text"])

Title: Biden plans to visit Texas, ask FEMA to accelerate major disaster declaration
Description: President Joe Biden said Friday that he plans to visit Texas next week as millions of residents there continue struggling with power outages after a major winter storm.Biden also said he will ask the administration of the Federal Emergent Management Agency to accelerate a request for a declaration of a major disaster in Texas, which would free up federal funds to aid in relief efforts there."As I said when I ran, I'm going to be a president for all Americans," said Biden, who was defeated last fall in Texas' presidential election by then-President Donald Trump."I'm, going to sign [that] declaration once that's in front of me" which "God-willing will bring relief to a lot of Texans," the president said.Biden said he had planned on traveling to Texas in the middle of next week, but cautioned that "I don't want to be a burden."He said that "if I can do it without creating a burden for folks, 

In [11]:
display(new_dataset.select(range(10)).to_pandas())

,title,url,published_at,author,publisher,short_description,keywords,header_image,raw_description,description,scraped_at,full_text
0,Santoli’s Wednesday market notes: Could Septem...,https://www.cnbc.com/2021/09/29/santolis-wedne...,2021-09-29T17:09:39+0000,Michael Santoli,CNBC,"This is the daily notebook of Mike Santoli, CN...","cnbc, Premium, Articles, Investment strategy, ...",https://image.cnbcfm.com/api/v1/image/10694960...,"<div class=""group""><p><em>This is the daily no...","This is the daily notebook of Mike Santoli, CN...",2021-10-30 14:11:23.709372,Title: Santoli’s Wednesday market notes: Could...
1,My take on the early Brexit winners and losers,https://www.cnbc.com/2016/06/24/ian-bremmers-t...,2016-06-24T13:50:48-0400,NaN,CNBC,This commentary originally ran on Facebook. Bo...,"Articles, Politics, Europe News, European Cent...",https://fm.cnbc.com/applications/cnbc.com/reso...,NaN,NaN,2021-10-30 14:11:23.820139,Title: My take on the early Brexit winners and...
2,Europe&#039;s recovery depends on Renzi&#039;s...,https://www.cnbc.com/2014/03/25/europes-recove...,2014-03-25T13:29:45-0400,NaN,CNBC,"In spring, ambitious reforms began in Italy. U...","Articles, Business News, Economy, Europe Econo...",https://fm.cnbc.com/applications/cnbc.com/reso...,NaN,NaN,2021-10-30 14:11:23.854710,Title: Europe&#039;s recovery depends on Renzi...
3,US Moves Closer to Becoming A Major Shareholde...,https://www.cnbc.com/2009/04/22/us-moves-close...,2009-04-22T19:49:03+0000,Michelle Caruso-Cabrera,CNBC,The US government is increasingly likely to co...,"cnbc, Articles, General Motors Co, Business Ne...",https://image.cnbcfm.com/api/v1/image/24947979...,"<div class=""group""><p>The US government is inc...",The US government is increasingly likely to co...,2021-10-30 14:11:24.261143,Title: US Moves Closer to Becoming A Major Sha...
4,Trump: 'Mission accomplished' on 'perfectly ex...,https://www.cnbc.com/2018/04/14/trump-mission-...,2018-04-14T14:59:04+0000,Javier E. David,CNBC,NaN,"cnbc, Articles, George W. Bush, Vladimir Putin...",https://image.cnbcfm.com/api/v1/image/10513177...,"<div class=""group""></div>,<div class=""group""><...",President Donald Trump hailed the U.S.-led int...,2021-10-30 14:11:24.489490,Title: Trump: 'Mission accomplished' on 'perfe...
5,Chevron CEO Watson says he supports Trump on t...,https://www.cnbc.com/2017/03/07/chevron-ceo-wa...,2017-03-07T23:07:14+0000,Lauren Thomas,CNBC,Chevron Chief Executive John Watson told CNBC ...,"cnbc, Articles, White House, Oil and Gas, Chev...",https://image.cnbcfm.com/api/v1/image/10322670...,"<div class=""group""><p><a href=""//www.cnbc.com/...",Chevron Chief Executive John Watson told CNBC ...,2021-10-30 14:11:24.736488,Title: Chevron CEO Watson says he supports Tru...
6,22. Hexadite,https://www.cnbc.com/2017/02/28/upstart-25-hex...,2017-02-28T11:00:37-0500,NaN,CNBC,"Founders: Eran Barak, Barak Klinghofer (chief ...","Articles, Technology, CNBC Upstart 2018, Bosto...",https://fm.cnbc.com/applications/cnbc.com/reso...,NaN,NaN,2021-10-30 14:11:24.797620,Title: 22. Hexadite\nDescription:
7,European stocks close higher on supportive Fed...,https://www.cnbc.com/2020/12/17/european-stock...,2020-12-17T06:08:38+0000,"Elliot Smith,Holly Ellyatt",CNBC,LONDON — European stocks closed higher on Thur...,"cnbc, Articles, World economy, World Markets, ...",https://image.cnbcfm.com/api/v1/image/10681242...,"<div class=""group""><p>LONDON — European stocks...",LONDON — European stocks closed higher on Thur...,2021-10-30 14:11:24.834045,Title: European stocks close higher on support...
8,"Fewer investors have a 'fear of missing out,' ...",https://www.cnbc.com/2019/07/26/fear-of-missin...,2019-07-28T21:00:47+0000,Stephanie Landsman,CNBC,Oppenheimer Asset Management's John Stoltzfus ...,"cnbc, Articles, S&P 500 Fut (Dec'21), Technolo...",https://image.cnbcfm.com/api/v1/image/10567396...,"<div class=""group""><p>Oppenheimer Asset Manage...",Oppenheimer Asset Management's John Stoltzfus ...,2021-10-3

In [12]:
from transformers import BitsAndBytesConfig


model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"Loading model: {model_id}")
print("This may take a few minutes...")


quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4")

model = AutoModelForCausalLM.from_pretrained(model_id,
                                             torch_dtype = 'auto',
                                             quantization_config=quantization_config,
                                             device_map = 'auto',
                                             trust_remote_code = True)

model.eval()
print("Model loaded in 4-bit successfully!")

Loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
This may take a few minutes...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded in 4-bit successfully!


In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [14]:
llm_pipeline = pipeline("text-generation",
                        model=model,
                        tokenizer=tokenizer,
                        torch_dtype='auto',
                        device_map='auto')

print("Test generation pipeline created successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Test generation pipeline created successfully!


In [15]:
test_question = "Explain what electric cars are in simple terms. Keep the thinking short."
test_prompt = f"<|im_start|>user\n{test_question}<|im_end|>\n<|im_start|>assistant\n<think>\n"
print(f"Test prompt:\n{test_prompt}")

Test prompt:
<|im_start|>user
Explain what electric cars are in simple terms. Keep the thinking short.<|im_end|>
<|im_start|>assistant
<think>



In [16]:
print("Testing model with a simple instruction")

outputs = llm_pipeline(test_prompt,
                       max_new_tokens=4000,
                       do_sample=True,
                       temperature=0.7,
                       top_p=0.9)

outputs

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing model with a simple instruction


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': "<|im_start|>user\nExplain what electric cars are in simple terms. Keep the thinking short.<|im_end|>\n<|im_start|>assistant\n<think>\nAlright, so I need to explain what electric cars are in simple terms. Let me start by breaking down the concept. Electric cars are designed to run on electricity, which is a big difference from how cars usually work. Cars have internal combustion engines that burn fuel to power the engine, which in turn spins the wheels. Electric cars, on the other hand, don't have that engine inside. Instead, they use generators or batteries to produce electricity.\n\nI should also mention that electric cars are often referred to as EVs, which stands for Electric Vehicle. They come in different types, like plug-in hybrid electric vehicles (PEHs), which have a battery and an internal combustion engine. Plug-in hybrid electric vehicles (PHEVs) are similar but might have a more efficient engine. Hybrid vehicles don't use a battery, so they need a separ

In [18]:
def format_model_output(output):

    # Extract the content within <think> tags as Reason
    reason_start = output.find("<think>") + len("<think>")
    reason_end = output.find("</think>")
    reason = output[reason_start:reason_end].strip()

    # Extract the content after </think> as Output
    output_start = reason_end + len("</think>")
    model_output_content = output[output_start:].strip()

    # Format the result
    reason = f"**Reason**:\n{reason}\n"
    output = f"**Output**:\n{model_output_content}"
    return reason, output

In [19]:
# Extract and print the generated text (only the response part)
full_output = outputs[0]["generated_text"]
reason, output = format_model_output(full_output)
print_markdown(reason + output)

**Reason**:
Alright, so I need to explain what electric cars are in simple terms. Let me start by breaking down the concept. Electric cars are designed to run on electricity, which is a big difference from how cars usually work. Cars have internal combustion engines that burn fuel to power the engine, which in turn spins the wheels. Electric cars, on the other hand, don't have that engine inside. Instead, they use generators or batteries to produce electricity.

I should also mention that electric cars are often referred to as EVs, which stands for Electric Vehicle. They come in different types, like plug-in hybrid electric vehicles (PEHs), which have a battery and an internal combustion engine. Plug-in hybrid electric vehicles (PHEVs) are similar but might have a more efficient engine. Hybrid vehicles don't use a battery, so they need a separate source of power, like a fuel cell or battery.

I should also touch on the advantages of electric cars. They don't require charging stations, which is great because charging can be expensive and time-consuming. They're also more environmentally friendly since they don't emit as much CO2. Plus, they can run on a single type of power source, which is convenient.

Wait, should I mention anything about charging methods? I think that's part of the EVs, so maybe I should include that. Some electric cars can charge internally, while others need a battery to power the charger. I should also note that electric cars are becoming more affordable, especially with cheaper batteries and more charging infrastructure.

I should keep the explanation short and simple, avoiding jargon. Maybe use bullet points or a numbered list to make it clear. Also, ensure that the explanation flows logically from the basics to the advantages, so the reader can understand the key points without getting lost.

I think I've covered the main points. Let me summarize to make sure I didn't miss anything.
**Output**:
**Electric Cars Explained Simply**

1. **Core Function**: Electric cars use electricity to power their wheels, eliminating the need for internal combustion engines.
   
2. **Types of Electric Cars**:
   - **Plug-in Hybrid Electric Vehicles (PEHs)**: Combine a battery with an internal combustion engine for efficiency.
   - **Plug-in Hybrid Electric Vehicles (PHEVs)**: Similar to PEHs but may have a more efficient engine.
   - **Hybrid Vehicles**: Do not use a battery; require a separate power source like a fuel cell.
   
3. **Advantages**:
   - **No Charging Stations**: Can run on a single power source without needing a battery.
   - **Environmental Benefits**: Lower CO2 emissions compared to internal combustion engines.
   - **Convenience**: More affordable and easier to use, especially with better batteries and charging infrastructure.

4. **Conclusion**: Electric cars offer a clean, efficient, and environmentally friendly alternative to traditional cars, making them a popular choice for modern consumers.

In [21]:
import re  # Import regular expressions for parsing

def analyze_news_sentiment(news_text, llm_pipeline):
    """
    Analyzes news sentiment using the LLM, providing reasoning and classification.

    Args:
        news_text (str): The combined title and description of the news.
        llm_pipeline (transformers.pipeline): The initialized text-generation pipeline.

    Returns:
        str: A string containing the reasoning and classification.
    """

    # Define the Prompt Template with specific instructions for the model
    test_question = f"""You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    {news_text}"""

    # Format the prompt according to DeepSeek's expected input format with thinking tags
    test_prompt = f"<|im_start|>user\n{test_question}<|im_end|>\n<|im_start|>assistant\n<think>\n"

    # Print the prompt for debugging purposes to verify what's being sent to the model
    print(f"Test prompt:\n{test_prompt}")

    # Run the model inference with specific generation parameters
    outputs = llm_pipeline(
        test_prompt,
        max_new_tokens = 4000,
        do_sample = True,
        temperature = 0.7,
        top_p = 0.9)

    # Extract the full generated text and parse it to separate reasoning from classification
    # The format_model_output function likely separates the thinking process from the final answer
    full_output = outputs[0]["generated_text"]
    reason, output = format_model_output(full_output)

    # Return both the reasoning process and the final sentiment classification
    return reason, output


In [22]:
# Print a separator line for clarity in the output
print("\n" + "=" * 30 + " TESTING ANALYSIS " + "=" * 30)


# Select a few random indices from the news dataset to test the analysis function
random_indices = random.sample(range(len(new_dataset)), 3)

# Iterate over each randomly selected index
for index in random_indices:
    # Retrieve the full text of the news item at the current index
    sample_news = new_dataset[index]["full_text"]

    # Analyze the sentiment of the sample news using the sentiment analysis function
    reason, output = analyze_news_sentiment(sample_news, llm_pipeline)

    # Print the analysis result header for the current index
    print(f"\n--- Analysis Result for Index {index} ---")

    # Display the reasoning and output in a formatted markdown style
    print_markdown(f"{reason}\n\n{output}")

    # Print a separator line for better readability between results
    print("-" * 60)

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



============================== TESTING ANALYSIS ==============================
Test prompt:
<|im_start|>user
You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    Title: Telecom Italia Reports 6.3% Decline in Net Profit
Description: Telecom Italia said Thursday that full-year net profit fell 6.3% on declining margins and tighter competition in the domestic wireline market.Italy's largest telecommunications operator said 2006 net profit stood at 3 billion euros ($3.9 billion) from 3.2 billion euros a year earlier. The result was slightly above the 2.96 billion euros ($3.88 billion) average estimate of

**Reason**:
Okay, so I'm looking at this news text about Telecom Italia. The title says they reported a 6.3% decline in net profit. That's pretty significant because it's a big operator in Italy, so it makes sense they'd be affected by market conditions.

Looking at the description, they mention full-year net profit fell 6.3% on declining margins and tighter competition. That's a strong indicator of the economy's performance affecting their profits. Italy is a key player, so this decline could have broader implications.

The company also reported revenue growth of 4.5% to 31.3 billion euros. That's an increase compared to the previous year's 29.9 billion euros. This shows they're doing well in terms of revenue, which is good for their finances.

They mentioned their former monopoly confirmed their dividend policy, proposing a 0.14 euro dividend. This is a positive sign because it shows they're committed to providing regular payouts, which can be a good indicator of investor confidence.

They also noted that Telecom Italia is coming off a management shakeup, which might affect their dividend policy. This means they're in a tricky position where both their dividend and profit performance could be affected.

The company reported net profit of 3 billion euros, which is slightly above their 2.96 billion euros average estimate. This suggests they're performing better than their expectations, but there's still some room for improvement.

Overall, the key points are the decline in net profit, revenue growth, dividend policy, and the comparison to industry averages. These points give a comprehensive overview of Telecom Italia's performance in the domestic wireline market during the year.


**Output**:
1. REASONING: Telecom Italia reported a 6.3% decline in net profit, reflecting broader economic conditions and tighter competition in the domestic wireline market. The company also reported revenue growth and maintained a dividend policy, though their net profit was slightly above industry average estimates.

2. SENTIMENT: Positive

3. TAG: Telecom Italia

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------
Test prompt:
<|im_start|>user
You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    Title: Economy Difficult, but Signs of Firming Seen: GE CEO
Description: The economy is still difficult, but we’re seeing some signs of firming, General Electric CEO Jeff Immelt told CNBC in an interview. "I think that while the economy is still difficult, we’re starting to firm," said Immelt. "We’re starting to see that the sequential appliance orders are firming, we’re starting to see some signs that businesses are ready to invest again. I think in certain 

**Reason**:
Alright, let's break this down. The user has provided a news text and some specific instructions to follow. First, I need to analyze the news text to extract the relevant information. The title mentions that the economy is still difficult but there are signs of firming. The description talks about GE CEO Jeff Immelt's comments, specifically pointing out that sequential appliance orders are firming and indicating some confidence in the system. Additionally, there's a reference to Cisco's CEO John Chambers, who agrees on the technology's potential for recovery, emphasizing business risks and the importance of smart electricity use.

Next, I need to extract the three requested items:
1. Reasoning: Brief financial reasoning. I should summarize the main points from Immelt and Chambers. It's important to highlight that while the economy is tough, the signs indicate a potential recovery, though not guaranteed.
2. Sentiment: Choose one: Positive, Negative, Neutral. Given the mixed environment and the signs pointing to recovery, it's a mix of positive and neutral. I'll go with Neutral.
3. Tag: A concise topic tag. The key aspects are the economy, recovery signs, GE, and tech. So, "Economy Recovery Signs GE Tech"

I also need to ensure that I don't add any extra text or explanations, just follow the structure as specified. Let me double-check to make sure I haven't missed any key points from the news text, especially regarding the economic indicators and the companies involved. Everything seems to be covered within the three items.


**Output**:
1. REASONING: While the economy is still difficult, the signs of recovery suggest a potential for recovery in certain sectors, though not guaranteed. This is a mix of positive and neutral sentiment.
2. SENTIMENT: Neutral
3. TAG: Economy Recovery Signs GE Tech

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------
Test prompt:
<|im_start|>user
You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    Title: Fitch cuts Turkey rating outlook to negative in wake of failed coup attempt
Description: Ratings agency Fitch affirmed its sovereign rating on Turkey at  'BBB-', the lowest investment grade, on Friday but lowered its outlook to negative from stable in a review after last  month's attempted military coup.   Investors have been rattled by both the failed July 15 putsch,  when a group of rogue soldiers attempted to overthrow the  government, and the wides

**Reason**:
Okay, so I'm looking at this news text about Fitch cutting the rating outlook for Turkey. The user wants me to analyze this and output three specific items: the reasoning, sentiment, and tag. Let me break this down step by step.

First, I need to understand the main points in the news text. Fitch is changing its outlook from stable to negative, which is a significant shift. They mentioned a failed coup attempt in July, which has been met with both public opposition and widespread repression. The economy has been affected by security issues in the tourism sector, and there are ongoing political pressures from the central bank.

Now, I need to figure out the sentiment. The user mentioned choosing between Positive, Negative, or Neutral. Given that Fitch is changing its outlook, which is a significant move, it's likely negative. The ratings agency is adjusting its outlook, which indicates a change in expectations.

Next, the tag. It should be a concise topic tag, one word. The news text is about Turkey's economy and political stability, so maybe "Economy and Political Stability" or something similar.

For the reasoning, I need to briefly explain why the sentiment is negative. Fitch's ratings often reflect the perceived quality of a company or country. If Turkey's economy is unstable, and there's political instability, that could lead to a negative rating outlook.

Putting it all together, I'll structure the response with each item labeled and prefixed appropriately.


**Output**:
1. REASONING: Fitch has adjusted its rating outlook for Turkey from stable to negative, reflecting a significant shift in expectations due to political instability and economic challenges.

2. SENTIMENT: Negative

3. TAG: Economy and Political Stability

------------------------------------------------------------


In [25]:
import gradio as gr  # Importing the Gradio library for creating the web interface
import random  # Importing the random library to generate random numbers
import re  # Importing the regular expressions library (not used in this snippet)
from transformers import pipeline  # Importing the pipeline function from transformers for model loading

# --- Gradio Helper Functions ---
def get_random_news():
    """Fetches and returns the full_text of a random news item."""
    if not new_dataset:  # Check if the news dataset is empty
        return "Error: No news items available."  # Return an error message if no news items are found
    random_index = random.randint(0, len(new_dataset) - 1)  # Generate a random index to select a news item
    news_text = new_dataset[random_index]['full_text']  # Retrieve the full text of the news item at the random index
    print(f"Fetched news item at index {random_index}")  # Print the index of the fetched news item
    return news_text  # Return the fetched news text

def perform_analysis(news_text):
    """Triggers analysis on the provided news text."""
    if not news_text or news_text.startswith("Error"):  # Check if the news text is empty or an error message
        return "Error: No news text to analyze."  # Return an error message if no valid news text is provided
    print(f"Analyzing news: {news_text[:50]}...")  # Print the first 50 characters of the news text being analyzed
    reason, output = analyze_news_sentiment(news_text, llm_pipeline)  # Analyze the sentiment of the news text
    return reason, output  # Return the reasoning and output from the analysis

# --- Build Gradio Interface ---
with gr.Blocks(theme=gr.themes.Glass()) as demo:  # Create a Gradio Blocks interface with a glass theme
    gr.Markdown("""
    # DeepSeek Financial News Analyzer
    Fetches a random news item from the dataset.
    Click 'Analyze News' to get sentiment classification and the model's reasoning.

    """)

    with gr.Row():  # Create a row for buttons
        btn_fetch = gr.Button("🔄 Fetch Random News Item")  # Button to fetch a random news item
        btn_analyze = gr.Button("💡 Analyze News", variant="primary")  # Button to analyze the news

    news_display = gr.Textbox(  # Textbox to display the news item
        label="📰 News Text (Title & Description)",  # Label for the textbox
        lines=8,  # Number of lines in the textbox
        interactive=False,  # Make the textbox non-interactive
        placeholder="Click 'Fetch Random News Item' to display news."  # Placeholder text
    )
    # Creates a collapsible panel
    with gr.Accordion("🤖 Model Reason", open=True):  # Accordion for model reasoning
        analysis_display = gr.Markdown()  # Markdown display for the reasoning

    with gr.Accordion("🤖 Model Output", open=True):  # Accordion for model output
        analysis_output = gr.Markdown()  # Markdown display for the output

    # --- Event Handlers ---
    btn_fetch.click(  # Set up click event for the fetch button
        fn=get_random_news,  # Function to call when the button is clicked
        inputs=[],  # No inputs needed
        outputs=[news_display]  # Output to the news display textbox
    )

    btn_analyze.click(  # Set up click event for the analyze button
        fn=perform_analysis,  # Function to call when the button is clicked
        inputs=[news_display],  # Input from the news display textbox
        outputs=[analysis_display, analysis_output]  # Outputs to the reasoning and output displays
    )

    # Load initial news item when the app starts
    demo.load(  # Load function to run when the app starts
        fn=get_random_news,  # Function to call to get a random news item
        inputs=None,  # No inputs needed
        outputs=[news_display]  # Output to the news display textbox
    )

# --- Launch the Gradio App ---
print("Launching Gradio demo...")  # Print message indicating the app is launching
demo.launch(debug=True)  # Launch the Gradio app in debug mode

C:\Users\princ\AppData\Local\Temp\ipykernel_39288\402503460.py:25: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Glass()) as demo:  # Create a Gradio Blocks interface with a glass theme


Launching Gradio demo...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Fetched news item at index 398


[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing news: Title: CEO Blog: Our Greatest New Threat
Descripti...
Test prompt:
<|im_start|>user
You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    Title: CEO Blog: Our Greatest New Threat
Description: I’m an engineer. I identify a problem and develop a solution. I’d like to think that the success of companies like Dyson – based in the UK– have helped the country in more than a few ways, including boosting its exports and tax revenues. Whether at home or abroad, politicians don’t really need me to identify new problems – they have enough of those. But where can I match my passion for a cause wit

...

Analyzing news: Title: Farmers & Merchants State Bank Announces Ne...
Test prompt:
<|im_start|>user
You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    Title: Farmers & Merchants State Bank Announces New Waterville Office Opening
Description: ARCHBOLD, Ohio, Oct. 1, 2012 /PRNewswire/ -- Paul S. Siebenmorgen, President and CEO of Farmers & Merchants State Bank in Archbold, Ohio, announced that construction will begin in the fall on the bank's twentieth office, in Waterville, Ohio.The full-service office will serve the Waterville and Whitehouse markets and include a drive-up ATM and three drive-up lan

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing news: Title: Google bought $750 million Lenovo stake on ...
Test prompt:
<|im_start|>user
You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    Title: Google bought $750 million Lenovo stake on January 30
Description: Internet search company Google Inc bought a 5.94 percent stake in China's Lenovo Group last month for $750 million, according to a disclosure on the Hong Kong stock exchange.Google acquired 618.3 million Lenovo shares at $1.213 per share on Jan. 30, the stock exchange said late on Thursday.Lenovo agreed to buy Google's Motorola handset division last week for $2.91 billion in a 

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing news: Title: US job cuts soar, thanks to tech layoffs: C...
Test prompt:
<|im_start|>user
You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    Title: US job cuts soar, thanks to tech layoffs: Challenger
Description: U.S. employers planned to cut nearly 50,000 positions in July, Challenger, Gray & Christmas reported Thursday, meting out 50 percent more job cuts than in the prior month.The global outplacement firm said in its monthly layoff report that planned payroll reductions hit 46,887, the second highest level of the year. That compares with June's 31,434 cuts, which were the fewest of 2